# Ön Eğitim ve Model Manzarası — eşlik eden notebook: alt kelime tokenleştirmesi

*Uygulamalı Doğal Dil İşleme — Token'lardan Ajanlara* kitabının 8. bölümü.
Önce bölümü okuyun: **[Ön Eğitim ve Model Manzarası](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/08-on-egitim.html)**.

Bu notebook bölümün tek bir kısmını kapsıyor: **[Alt kelime tokenleştirmesi: model metni nasıl görür](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/08-on-egitim.html#sec-alt-kelime)**.
Bu kısım önceden kitabın ikinci bölümündeydi; bu tokenizer'ları kullanan ön eğitimli
modellerin yanına taşındı ve hücreleri de onunla birlikte geldi. Bölümün geri kalanı (veri
hunisi, hesap aritmetiği, model manzarası) için henüz notebook hücresi yok.

Bölümün durum notu, bu kısımdaki her sayının ve kod çıktısının kod çalıştırılarak
üretildiğini söylüyor. Bu notebook o kodu, kısmın sırasıyla çalıştırıyor:

1. **Sıfırdan byte-pair encoding**: dört kelimelik bir derlemde birleştirmeleri öğrenmek ve
   hiç görülmemiş bir kelimede yeniden oynatmak.
2. **Karakter mi bayt mı**: BPE'nin neyle başladığı, UTF-8, bir karakterin ortasından geçen
   bayt düzeyi token'lar ve byte fallback.
3. **Türkçe için faturayı kim ödüyor**: OpenAI tokenizer'larında iki dilde aynı metin, ekleri
   üst üste binen bir Türkçe kelime ve gerçek bir tokenizer'ın dört aşaması.
4. **Karakter, alt kelime, bütün kelime**, küçük bir derlemde; ardından WordPiece ile
   Unigram LM üzerine derinleşme.
5. **WordPiece ne zaman `[UNK]` der**, bölümdeki mülakat cevabından.

**Sayılar nasıl okunur.** Aşağıda basılan her şey *bu koşunun* ürettiği şey. Bölüm aynı
çıktıyı basıyorsa hücre kitabın değerini yanına yazıyor ya da basılan bloğun tamamını
satır satır karşılaştırıyor ve ikisinin tutup tutmadığını söylüyor. Bölümün çıktıyı basıp
kodu basmadığı yerlerde buradaki kod, metni bizim okuyuşumuz; hücre bunu da söylüyor.

**Dil hakkında bir not.** Anlatım ve kod yorumları Türkçe; kodun kendisi ve *basılan*
etiketler İngilizce. Bunun sebebi, Türkçe bölümlerin kendi `text` bloklarının da İngilizce
olması — böylece burada bastığınız tabloyla kitaptaki tablo yan yana okunabiliyor.

**Etkileşimli şekiller.** Bölümdeki BPE birleştirme oynatıcısı bir Observable bileşeni ve
bir notebook'ta çalışamıyor; ilk kısmın hücreleri aynı birleştirmeleri basıyor.

**Süre.** İlk indirmeden sonra bir dizüstü CPU'sunda birkaç saniye. İndirilenler: iki
`tiktoken` sözlüğü (toplam yaklaşık 5 MB) ve Hugging Face Hub'dan üç tokenizer dosyası
(toplam yaklaşık 13 MB). Hiçbir hücre ağır değil, dolayısıyla `ANLP_SKIP_HEAVY=1` burada
bir şey değiştirmiyor.

## 0. Kurulum

İlk hücre yorumlayıcı ve paket sürümlerini kaydediyor. Bölümün durum notu `tiktoken` 0.13,
`sentencepiece` 0.2.2 ve `tokenizers` 0.22 diyor; kitabın kendi ölçümleri Python 3.14.5 üzerinde koştu
([Ek A](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/appendices/a-kurulum.html)). Sizin sayılarınız burada kayıtlı olanlardan
farklıysa önce bu hücreyi karşılaştırın.

İndirilenler, notebook deposunun kökündeki `data/cache/` altında önbelleğe alınıyor; git
orayı yoksayıyor, dolayısıyla ikinci koşu diskten okuyor. Depo dışında çalıştırırsanız
notebook onları kendi yanındaki `data/cache/` altına alıyor.

In [1]:
import io
import math
import os
import platform
import re
import sys
import time
import unicodedata
from collections import Counter
from importlib.metadata import version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

here = Path.cwd()
repo_root = next((p for p in [here, *here.parents] if (p / "scripts" / "check-structure.py").exists()), here)
CACHE = repo_root / "data" / "cache"
os.environ.setdefault("TIKTOKEN_CACHE_DIR", str(CACHE / "tiktoken"))  # tokenizer sözlükleri
DATA_HOME = str(CACHE / "scikit_learn_data")                           # 20 Newsgroups arşivi
os.environ.setdefault("HF_HOME", str(CACHE / "huggingface"))          # Hub'dan gelen tokenizer dosyaları
os.environ.setdefault("HF_HUB_VERBOSITY", "error")

print(f"Python        {platform.python_version()}  ({platform.system()} {platform.machine()})")
for pkg in ["numpy", "tiktoken", "sentencepiece", "tokenizers", "matplotlib"]:
    print(f"{pkg:<13} {version(pkg)}")

Python        3.12.13  (Darwin arm64)
numpy         2.5.2
tiktoken      0.13.0
sentencepiece 0.2.2
tokenizers    0.22.2
matplotlib    3.11.2


Karşılaştırmayı iki yardımcı yapıyor. `vs_book` bir sayıyı kitabınkinin yanına, kitabın
bastığı hassasiyette koyuyor. `record()` bir hücrenin bastığı her şeyin kopyasını tutmaya
başlıyor ve `compare_with_book(...)` o kopyayı bölümün bastığı çıktı bloğuyla
karşılaştırıyor. Her karakter tutuyorsa *line for line* diyor; değerler tutup sütunlar
farklı doldurulmuşsa *only the spacing differs* diyor (kitap bazen NumPy dizilerini
dolgusu alınmış hâlde basıyor).

In [2]:
def vs_book(label, ours, book, fmt="{:,}"):
    """Print this run's value beside the book's, compared at the book's printed precision."""
    ours_s, book_s = fmt.format(ours), fmt.format(book)
    verdict = "same as the book" if ours_s == book_s else "DIFFERS from the book"
    print(f"{label:<38} this run {ours_s:>9}   book {book_s:>9}   {verdict}")


class _Tee(io.TextIOBase):
    """Writes to the notebook as usual and keeps a copy."""

    def __init__(self, stream):
        self.stream, self.copy = stream, io.StringIO()

    def write(self, s):
        self.copy.write(s)
        self.stream.write(s)
        return len(s)

    def flush(self):
        self.stream.flush()


def record():
    """Start keeping a copy of everything printed, to compare with the book's output."""
    if isinstance(sys.stdout, _Tee):
        sys.stdout = sys.stdout.stream
    sys.stdout = _Tee(sys.stdout)


def _squash(lines):
    """Collapse runs of spaces, and the padding NumPy puts inside brackets."""
    return [re.sub(r"\[ +| +\]", lambda m: m.group().strip(), " ".join(line.split())) for line in lines]


def compare_with_book(book_text):
    """Stop recording, and compare what was printed with the output block the book prints."""
    if not isinstance(sys.stdout, _Tee):
        raise RuntimeError("call record() before the code whose output you want to compare")
    ours_text = sys.stdout.copy.getvalue()
    sys.stdout = sys.stdout.stream
    ours = [line.rstrip() for line in ours_text.strip("\n").splitlines()]
    book = [line.rstrip() for line in book_text.strip("\n").splitlines()]
    print()
    if ours == book:
        print(f"-> same as the book, line for line ({len(book)} lines)")
    elif _squash(ours) == _squash(book):
        print(f"-> same values as the book ({len(book)} lines); only the spacing differs")
    else:
        print("-> DIFFERS from the book:")
        for k in range(max(len(ours), len(book))):
            o = ours[k] if k < len(ours) else "(no line)"
            b = book[k] if k < len(book) else "(no line)"
            if " ".join(o.split()) != " ".join(b.split()):
                print(f"   line {k + 1}\n     this run: {o}\n     book    : {b}")

## 1. Sıfırdan byte-pair encoding

[Bölümdeki kısım](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/08-on-egitim.html#sec-metin-veri-bpe). Bölüm algoritmadan önce, hiç görülmemiş bir
kelimeyi üç birim boyutunda gösteriyor. Bütün kelime olarak `<UNK>` oluyor. Karakter olarak
hiçbir şey kaybolmuyor, ama kelime uzuyor ve hiçbir birim tek başına bir şey söylemiyor.
Alt kelime olarak ise modelin başka kelimelerden tanıdığı üç parçaya bölünüyor. Türkçe
baskı bunun için `evlerden` kelimesini, İngilizce baskı `unbreakable` kelimesini kullanıyor.
Bölüm, alt kelime satırının `cl100k_base` tokenizer'ının gerçekten yaptığı bölünme olduğunu
söylüyor; hücre bunu iki kelime için de kontrol ediyor.

`tiktoken`'ın ilk kullanımı kodlamanın sözlük dosyasını indiriyor; nereye kaydedildiğini
üçüncü kısım söylüyor.

In [3]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")
for word, book in [("unbreakable", "un break able"), ("evlerden", "ev ler den")]:
    print(f"{word}: as characters, {len(word)} units: {' '.join(word)}")
    vs_book("  as subwords (cl100k_base)", " ".join(enc.decode([t]) for t in enc.encode(word)), book, "{}")

unbreakable: as characters, 11 units: u n b r e a k a b l e
  as subwords (cl100k_base)            this run un break able   book un break able   same as the book
evlerden: as characters, 8 units: e v l e r d e n
  as subwords (cl100k_base)            this run ev ler den   book ev ler den   same as the book


Şimdi bu parçaları bulan algoritma. Karakterlerle başlayın, her komşu çifti sayın, en sık
çifti yeni bir sembolde birleştirin, tekrarlayın. Kod bölümün kodu, satır satır.

**Neye bakacaksınız:** ilk üç birleştirme `est</w>`'yi kuruyor — İngilizce üstünlük eki ve
kelime sınırı — ve algoritmaya İngilizcenin ek aldığını kimse söylemedi.

In [4]:
record()
from collections import Counter

corpus = {"low": 5, "lower": 2, "newest": 6, "widest": 3}
words = {tuple(w) + ("</w>",): n for w, n in corpus.items()}

def pair_counts(words):
    pairs = Counter()
    for symbols, freq in words.items():
        for a, b in zip(symbols, symbols[1:]):
            pairs[(a, b)] += freq
    return pairs

def merge(words, pair):
    a, b = pair
    out = {}
    for symbols, freq in words.items():
        new, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                new.append(a + b)
                i += 2
            else:
                new.append(symbols[i])
                i += 1
        out[tuple(new)] = freq
    return out

merges = []
for step in range(1, 11):
    best, count = pair_counts(words).most_common(1)[0]
    merges.append(best)
    words = merge(words, best)
    print(f"merge {step:>2}: {best[0]!r} + {best[1]!r} -> "
          f"{best[0] + best[1]!r}  (seen {count} times)")
compare_with_book("""
merge  1: 'e' + 's' -> 'es'  (seen 9 times)
merge  2: 'es' + 't' -> 'est'  (seen 9 times)
merge  3: 'est' + '</w>' -> 'est</w>'  (seen 9 times)
merge  4: 'l' + 'o' -> 'lo'  (seen 7 times)
merge  5: 'lo' + 'w' -> 'low'  (seen 7 times)
merge  6: 'n' + 'e' -> 'ne'  (seen 6 times)
merge  7: 'ne' + 'w' -> 'new'  (seen 6 times)
merge  8: 'new' + 'est</w>' -> 'newest</w>'  (seen 6 times)
merge  9: 'low' + '</w>' -> 'low</w>'  (seen 5 times)
merge 10: 'w' + 'i' -> 'wi'  (seen 3 times)
""")

merge  1: 'e' + 's' -> 'es'  (seen 9 times)
merge  2: 'es' + 't' -> 'est'  (seen 9 times)
merge  3: 'est' + '</w>' -> 'est</w>'  (seen 9 times)
merge  4: 'l' + 'o' -> 'lo'  (seen 7 times)
merge  5: 'lo' + 'w' -> 'low'  (seen 7 times)
merge  6: 'n' + 'e' -> 'ne'  (seen 6 times)
merge  7: 'ne' + 'w' -> 'new'  (seen 6 times)
merge  8: 'new' + 'est</w>' -> 'newest</w>'  (seen 6 times)
merge  9: 'low' + '</w>' -> 'low</w>'  (seen 5 times)
merge 10: 'w' + 'i' -> 'wi'  (seen 3 times)



-> same as the book, line for line (10 lines)


Token'lara ayırmak için öğrenilen birleştirmeler, öğrenildikleri sırayla tekrar oynatılıyor.
`lowest` eğitimde hiç geçmedi. Bu hücre aynı zamanda bölümün etkileşimli BPE aracının notebook'taki
karşılığı: listeye kendi kelimelerinizi ekleyip yeniden çalıştırın.

In [5]:
record()
def apply_merges(word, merges):
    symbols = tuple(word) + ("</w>",)
    for pair in merges:
        symbols = tuple(merge({symbols: 1}, pair).keys())[0]
    return symbols

for w in ["lowest", "newer", "slow", "widest"]:
    print(f"{w:<8} -> {' '.join(apply_merges(w, merges))}")
compare_with_book("""
lowest   -> low est</w>
newer    -> new e r </w>
slow     -> s low</w>
widest   -> wi d est</w>
""")

lowest   -> low est</w>
newer    -> new e r </w>
slow     -> s low</w>
widest   -> wi d est</w>



-> same as the book, line for line (4 lines)


**Çekince.** Birleştirme tablosu *karakterler* üzerinde öğrenildi ve gördüğü alfabe
yalnızca `low`, `lower`, `newest` ve `widest` kelimelerinin harflerinden ibaretti.
Dolayısıyla görülmemiş bir **karakter** onu hâlâ kırıyor. Hücre bu küçük tokenizer'ın
bildiği sembolleri (başlangıç sembolleri artı on birleştirme) listeliyor ve `çay`ın her
parçasını onlara karşı kontrol ediyor.

In [6]:
pieces = apply_merges("çay", merges)
vs_book('apply_merges("çay", merges)', repr(pieces), "('ç', 'a', 'y', '</w>')", "{}")

known = {s for w in corpus for s in tuple(w) + ("</w>",)} | {a + b for a, b in merges}
print(f"\nsymbols the toy tokenizer knows ({len(known)}): {sorted(known)}")
for p in pieces:
    print(f"  {p!r:<7} {'known' if p in known else 'never seen in training'}")

apply_merges("çay", merges)            this run ('ç', 'a', 'y', '</w>')   book ('ç', 'a', 'y', '</w>')   same as the book

symbols the toy tokenizer knows (21): ['</w>', 'd', 'e', 'es', 'est', 'est</w>', 'i', 'l', 'lo', 'low', 'low</w>', 'n', 'ne', 'new', 'newest</w>', 'o', 'r', 's', 't', 'w', 'wi']
  'ç'     never seen in training
  'a'     never seen in training
  'y'     never seen in training
  '</w>'  known


**Nasıl okunur.** `ç`, `a` ve `y` eğitim kelimelerinde hiç geçmedi, dolayısıyla hiçbir
birleştirme onlara dokunamıyor. Dördüncü parça, yordamın her kelimeye eklediği kelime sonu
işareti `</w>`. Bayt düzeyinde çalışan tokenizer'lar bu açığı, karakterler yerine UTF-8'in
256 olası **baytından** başlayarak kapatıyor; nasıl yaptıklarını ikinci kısım gösteriyor.

## 2. Karakter mi bayt mı: BPE neyle başlar

[Bölümdeki kısım](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/08-on-egitim.html#sec-metin-veri-baytlar). Önce dört kavramı birbirinden ayırmak
gerekiyor. *Karakter*, ekranda gördüğünüz harf ya da işaret. Bilgisayar yalnızca sayı
saklar, bu yüzden her karaktere bir numara gerekir. *ASCII* (1963) bunlardan 128 tane
içerir: İngilizce harfler, rakamlar ve noktalama. *Unicode* aynı fikri bütün yazı
sistemlerine genişletir; bu numaraya karakterin *kod noktası* denir. *Bayt* 0 ile 255
arasında bir sayı tutar ve *UTF-8* kodlaması her kod noktasını bir ile dört bayt arasında
yazar.

Bölüm bunların hepsini `çay` kelimesi için küçük bir tabloda topluyor. Hücre tabloyu
yeniden kuruyor ve sayılarını kontrol ediyor.

**Neye bakacaksınız:** üç karakter, üç kod noktası, dört bayt.

In [7]:
shown = "çay"
print(f"{'':<22}" + "".join(f"{ch:>15}" for ch in shown))
print(f"{'Unicode code point':<22}" + "".join(f"{f'{ord(ch)} (U+{ord(ch):04X})':>15}" for ch in shown))
print(f"{'in the ASCII table?':<22}" + "".join(f"{'yes' if ord(ch) < 128 else 'no':>15}" for ch in shown))
print(f"{'UTF-8 bytes':<22}" + "".join(f"{', '.join(str(b) for b in ch.encode('utf-8')):>15}" for ch in shown))
print()
vs_book('characters: len("çay")', len(shown), 3)
vs_book('bytes: len("çay".encode("utf-8"))', len(shown.encode("utf-8")), 4)
vs_book("code point of ç", ord("ç"), 231)
vs_book("UTF-8 bytes of ç", list("ç".encode("utf-8")), [195, 167], "{}")

                                    ç              a              y
Unicode code point       231 (U+00E7)    97 (U+0061)   121 (U+0079)
in the ASCII table?                no            yes            yes
UTF-8 bytes                  195, 167             97            121

characters: len("çay")                 this run         3   book         3   same as the book
bytes: len("çay".encode("utf-8"))      this run         4   book         4   same as the book
code point of ç                        this run       231   book       231   same as the book
UTF-8 bytes of ç                       this run [195, 167]   book [195, 167]   same as the book


UTF-8, bir ASCII harfi için bir bayt, ASCII dışındaki Türkçe harfler için iki, Çince
karakterlerin çoğu için üç, emojilerin çoğu için dört bayt kullanıyor. Kod bölümün kodu.

In [8]:
record()
for ch in ["a", "ç", "ş", "İ", "世", "😀"]:
    b = ch.encode("utf-8")
    print(f"{ch}  U+{ord(ch):04X}  {len(b)} byte(s): {' '.join(f'{x:02X}' for x in b)}")
compare_with_book("""
a  U+0061  1 byte(s): 61
ç  U+00E7  2 byte(s): C3 A7
ş  U+015F  2 byte(s): C5 9F
İ  U+0130  2 byte(s): C4 B0
世  U+4E16  3 byte(s): E4 B8 96
😀  U+1F600  4 byte(s): F0 9F 98 80
""")

a  U+0061  1 byte(s): 61
ç  U+00E7  2 byte(s): C3 A7
ş  U+015F  2 byte(s): C5 9F
İ  U+0130  2 byte(s): C4 B0
世  U+4E16  3 byte(s): E4 B8 96
😀  U+1F600  4 byte(s): F0 9F 98 80



-> same as the book, line for line (6 lines)


**Baytlardan başlamak.** Baytlardan başlayan bir tokenizer'ın alfabesi baştan tamdır, çünkü
her dildeki her metin 256 bayt değerinin bir dizisidir. Aşağıda OpenAI'ın bayt düzeyindeki
`cl100k_base` kodlaması, her token'ın arkasındaki baytları gösteriyor. Kod bölümün kodu.

**Neye bakacaksınız:** `ç` tek token, `ŞİŞE` `şişe`'nin iki katına mal oluyor ve hiçbir şey
`<UNK>` olmuyor.

In [9]:
record()
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")
for text in ["çay", "şişe", "ŞİŞE", "😀"]:
    pieces = [enc.decode_single_token_bytes(t) for t in enc.encode(text)]
    print(f"{text:<5} {len(pieces)} tokens  {pieces}")
compare_with_book("""
çay   2 tokens  [b'\\xc3\\xa7', b'ay']
şişe  3 tokens  [b'\\xc5\\x9fi', b'\\xc5\\x9f', b'e']
ŞİŞE  6 tokens  [b'\\xc5', b'\\x9e', b'\\xc4\\xb0', b'\\xc5', b'\\x9e', b'E']
😀     2 tokens  [b'\\xf0\\x9f\\x98', b'\\x80']
""")

çay   2 tokens  [b'\xc3\xa7', b'ay']
şişe  3 tokens  [b'\xc5\x9fi', b'\xc5\x9f', b'e']
ŞİŞE  6 tokens  [b'\xc5', b'\x9e', b'\xc4\xb0', b'\xc5', b'\x9e', b'E']
😀     2 tokens  [b'\xf0\x9f\x98', b'\x80']



-> same as the book, line for line (4 lines)


**Ama baytlar karakterlere saygı duymuyor.** Bölümün düzeltmesi: bayt düzeyinde BPE sık
geçen *bayt* dizilerini birleştiriyor ve birleştirmeler Unicode karakter sınırlarına bağlı
değil. Sık geçen karakterlerin çoğu gerçekten tek token oluyor, ama bir token birkaç
karakteri de, bir karakterin yalnızca bir parçasını da temsil edebiliyor. Aşağıdaki hücre
her token'ın baytlarını tek başına çözüyor ve o baytların geçerli UTF-8 olup olmadığını
bildiriyor.

In [10]:
import tiktoken

def boundary_report(enc_name, strings):
    enc = tiktoken.get_encoding(enc_name)
    print(f"{enc_name}")
    for s in strings:
        parts = [enc.decode_single_token_bytes(t) for t in enc.encode(s)]
        kind = []
        for b in parts:
            try:
                b.decode("utf-8")
                kind.append("whole")
            except UnicodeDecodeError:
                kind.append("PARTIAL")
        print(f"  {s:<22} {len(parts):>2} tokens  {kind}")
        print(f"  {'':<22}    {parts}")
    print()


for name in ["cl100k_base", "o200k_base"]:
    boundary_report(name, ["çay", "😀", "🇹🇷", "世界"])

cl100k_base
  çay                     2 tokens  ['whole', 'whole']
                            [b'\xc3\xa7', b'ay']
  😀                       2 tokens  ['PARTIAL', 'PARTIAL']
                            [b'\xf0\x9f\x98', b'\x80']
  🇹🇷                      6 tokens  ['PARTIAL', 'PARTIAL', 'PARTIAL', 'PARTIAL', 'PARTIAL', 'PARTIAL']
                            [b'\xf0\x9f', b'\x87', b'\xb9', b'\xf0\x9f', b'\x87', b'\xb7']
  世界                      3 tokens  ['PARTIAL', 'PARTIAL', 'whole']
                            [b'\xe4\xb8', b'\x96', b'\xe7\x95\x8c']

o200k_base
  çay                     2 tokens  ['whole', 'whole']
                            [b'\xc3\xa7', b'ay']
  😀                       1 tokens  ['whole']
                            [b'\xf0\x9f\x98\x80']
  🇹🇷                      4 tokens  ['PARTIAL', 'PARTIAL', 'PARTIAL', 'PARTIAL']
                            [b'\xf0\x9f\x87', b'\xb9', b'\xf0\x9f\x87', b'\xb7']
  世界                      1 tokens  ['whole']
                    

**Nasıl okunur.** `çay` iki token'a bölünüyor ve ikisi de geçerli UTF-8 — Türkçe `ç`
gerçekten tek token olmuş, ki sık bir karakter için olağan olan bu. Emoji ise karşı örnek:
`cl100k_base` altında `😀` iki token ve *hiçbiri* tek başına geçerli bir karakter değil;
birleştirme tablosu dört baytlık bir dizinin ortasından kesmiş. Türk bayrağı `🇹🇷` böyle
altı parça. Daha sonra ve daha büyük bir sözlükle eğitilen `o200k_base` `😀`'yı bütün
tutuyor ama bayrağı hâlâ bölüyor.

"Sık geçen bayt çiftleri önce birleşip tam karakterlere dönüşür" cümlesinin neden yanlış
bir zihinsel model olduğu bu. Algoritmanın hiçbir yerinde karakterin nerede bittiği
bilinmiyor. Pratikte şurada canınızı yakıyor: bir token dizisini keserken, prompt'u
rastgele bir token'dan bölmek elinizde yarım bir karakter bırakabilir.

**Byte fallback.** İki seçeneği birleştiren üçüncü bir tasarım. Sözlük karakterlerden
kurulur, ama içine 256 bayt değerinin her biri için birer token eklenir. Tokenizer
tanımadığı bir karakterle karşılaşınca `<UNK>` yazmak yerine o karakterin UTF-8 baytlarını
bu token'larla yazar. Bölüm, SentencePiece'i içinde hiç `ç` geçmeyen sekiz Türkçe kelimeyle
iki kez eğitiyor: bir kez bu seçenek kapalı, bir kez açık. Sonra ikisine de `çay` kelimesini
veriyor.

Kod bölümün kodu, tek bir farkla: bölüm üç küçük dosyasını çalışma dizinine yazıyor, bu
hücre ise depoya karışmasınlar diye `data/cache/` altına yazıyor.

**Neye bakacaksınız:** her satırın sonu, yani token'ların yeniden metne çevrilmiş hâli.

In [11]:
record()
import sentencepiece as spm

mini = CACHE / "sentencepiece-mini"                 # kitapta yok: dosyaları önbellekte tut
mini.mkdir(parents=True, exist_ok=True)

words = ["ev", "evde", "evden", "evler", "yol", "yolda", "yoldan", "yollar"]
with open(mini / "mini.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(words * 5) + "\n")            # bu metinde hiç 'ç' yok

for fallback in [False, True]:
    spm.SentencePieceTrainer.train(
        input=str(mini / "mini.txt"), model_prefix=str(mini / "mini"), model_type="bpe",
        vocab_size=20 + (256 if fallback else 0), byte_fallback=fallback,
        character_coverage=1.0, hard_vocab_limit=False, minloglevel=2,
        bos_id=-1, eos_id=-1, pad_id=-1)
    sp = spm.SentencePieceProcessor(model_file=str(mini / "mini.model"))
    ids = sp.encode("çay")
    print(f"byte_fallback={fallback!s:<5}  vocabulary {sp.get_piece_size():>3}  "
          f"{sp.encode('çay', out_type=str)}  ->  {sp.decode(ids)!r}")
compare_with_book("""
byte_fallback=False  vocabulary  20  ['▁', 'ç', 'a', 'y']  ->  ' ⁇ ay'
byte_fallback=True   vocabulary 276  ['▁', '<0xC3>', '<0xA7>', 'a', 'y']  ->  'çay'
""")

byte_fallback=False  vocabulary  20  ['▁', 'ç', 'a', 'y']  ->  ' ⁇ ay'
byte_fallback=True   vocabulary 276  ['▁', '<0xC3>', '<0xA7>', 'a', 'y']  ->  'çay'



-> same as the book, line for line (2 lines)


**Nasıl okunur.** Seçenek kapalıyken `ç` "bilinmeyen" kimliğini alıyor ve geri çevirince
yerinde `⁇` kalıyor: harf kayboldu. Açıkken sözlük 256 bayt token'ı kadar büyümüş (20'den
276'ya) ve `ç`, `<0xC3>` `<0xA7>` olarak yazılıyor. Bunlar yukarıdaki `çay` tablosundaki iki
bayt: onaltılık C3 = 195, A7 = 167. Metne geri çevirince `çay` eksiksiz dönüyor. Baştaki
`▁`, kelimenin başladığı yeri gösteren işaret; üçüncü kısım ona geri dönüyor.

## 3. Türkçe için faturayı kim ödüyor

[Bölümdeki kısım](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/08-on-egitim.html#sec-metin-veri-turkce). İnsan Hakları Evrensel Bildirgesi'nin 1.
maddesi, resmî İngilizce ve Türkçe metinleriyle (Birleşmiş Milletler yayını) OpenAI'ın iki
tokenizer'ından geçiriliyor. Kod bölümün kodu. Uzunluklar Unicode kod noktası olarak
sayılıyor; Python'da `len` bir metin için bunu döndürür.

İlk çağrı her kodlamanın sözlük dosyasını OpenAI'ın herkese açık blob deposundan indiriyor
ve `data/cache/tiktoken` altında önbelleğe alıyor. Bölüm bu kısımdaki her ölçümün Unicode
NFC normalleştirmesinden sonra alındığını söylüyor; hücre de iki metnin zaten NFC
olduğunu kontrol ediyor.

In [12]:
record()
import tiktoken

en = ("All human beings are born free and equal in dignity and rights. "
      "They are endowed with reason and conscience and should act towards "
      "one another in a spirit of brotherhood.")
tr = ("Bütün insanlar hür, haysiyet ve haklar bakımından eşit doğarlar. "
      "Akıl ve vicdana sahiptirler ve birbirlerine karşı kardeşlik "
      "zihniyeti ile hareket etmelidirler.")

for name in ["cl100k_base", "o200k_base"]:
    enc = tiktoken.get_encoding(name)
    a, b = len(enc.encode(en)), len(enc.encode(tr))
    print(f"{name}:")
    print(f"  english : {len(en):>4} code points -> {a:>3} tokens  "
          f"({len(en) / a:4.2f} code points/token)")
    print(f"  turkish : {len(tr):>4} code points -> {b:>3} tokens  "
          f"({len(tr) / b:4.2f} code points/token)")
    print(f"  Turkish/English token count ratio: {b / a:4.2f}")
compare_with_book("""
cl100k_base:
  english :  170 code points ->  33 tokens  (5.15 code points/token)
  turkish :  160 code points ->  60 tokens  (2.67 code points/token)
  Turkish/English token count ratio: 1.82
o200k_base:
  english :  170 code points ->  33 tokens  (5.15 code points/token)
  turkish :  160 code points ->  46 tokens  (3.48 code points/token)
  Turkish/English token count ratio: 1.39
""")
print("both strings already NFC:", unicodedata.is_normalized("NFC", en) and unicodedata.is_normalized("NFC", tr))

cl100k_base:
  english :  170 code points ->  33 tokens  (5.15 code points/token)
  turkish :  160 code points ->  60 tokens  (2.67 code points/token)
  Turkish/English token count ratio: 1.82
o200k_base:
  english :  170 code points ->  33 tokens  (5.15 code points/token)
  turkish :  160 code points ->  46 tokens  (3.48 code points/token)
  Turkish/English token count ratio: 1.39



-> same as the book, line for line (8 lines)
both strings already NFC: True


**Bir Türkçe kelime, dört ek.** Bölüm, ekler üst üste binerken tokenizer'ın `ev`i nasıl
böldüğünü basıyor ama kodu basmıyor. `' ev'`in başındaki boşluk, kelimenin bir boşluktan
sonra — yani cümle ortasında görüneceği gibi — kodlandığını söylüyor. **Bizim okuyuşumuz:**
`cl100k_base`, her kelimenin önünde bir boşluk. Hücre ayrıca bölümün üç token dediği
İngilizce çeviriyi, *from our houses*, sayıyor ve bölümün şu notunu kontrol ediyor: daha
yeni `o200k_base`, `imiz` parçasını bölmeden tutuyor.

In [13]:
enc = tiktoken.get_encoding("cl100k_base")

record()
for w in ["ev", "evler", "evlerimiz", "evlerimizden"]:
    ids = enc.encode(" " + w)
    print(f"{w:<14} {len(ids)} tokens  {[enc.decode([i]) for i in ids]}")
compare_with_book("""
ev             1 tokens  [' ev']
evler          2 tokens  [' ev', 'ler']
evlerimiz      4 tokens  [' ev', 'ler', 'im', 'iz']
evlerimizden   5 tokens  [' ev', 'ler', 'im', 'iz', 'den']
""")
print()
vs_book('tokens in " from our houses"', len(enc.encode(" from our houses")), 3)
o200k = tiktoken.get_encoding("o200k_base")
vs_book("o200k_base splits evlerimizden as",
        "|".join(o200k.decode([i]) for i in o200k.encode(" evlerimizden")).strip(), "ev|ler|imiz|den", "{}")

ev             1 tokens  [' ev']
evler          2 tokens  [' ev', 'ler']
evlerimiz      4 tokens  [' ev', 'ler', 'im', 'iz']
evlerimizden   5 tokens  [' ev', 'ler', 'im', 'iz', 'den']



-> same as the book, line for line (4 lines)

tokens in " from our houses"           this run         3   book         3   same as the book
o200k_base splits evlerimizden as      this run ev|ler|imiz|den   book ev|ler|imiz|den   same as the book


**Nasıl okunur.** `ev|ler|im|iz|den` dört sınırının üçünde gerçek biçimbilimle örtüşüyor.
Bölümün çekince koyduğu yer `im|iz`: *-imiz* tek bir biçimbirim ve BPE onu ikiye bölmüş,
çünkü `im` ve `iz` ayrı ayrı sık geçiyor. `o200k_base` bu kesimi yapmıyor; bu da kesimin
dile değil, birleştirme tablosuna bağlı olduğunu gösteriyor.

**Boşluklar nereye gitti?** Yukarıdaki ilk token `' ev'`, yani başında bir boşluk var.
GPT'nin BPE tokenizer'ları ve BERT'in WordPiece'i metni önce boşluk ve noktalamadan
kelimelere böler (*ön tokenleştirme*), alt kelimelere bölmeyi ancak ondan sonra, her
kelimenin içinde yapar. Bu yüzden hiçbir token iki kelimeye birden yayılmaz. Token'ları
yeniden metne çevirebilmek için kelimenin nerede başladığının token'ın üzerine yazılması
gerekir ve üç tokenizer ailesi bunu üç farklı işaretle yapıyor. Bölümün tablosu aynı iki
kelimeyi, `evlerimizden geldik`, üçünde de gösteriyor ama kodunu vermiyor. **Bizim
okuyuşumuz:** tabloda adı geçen üç tokenizer; boşluk, tablodaki gibi `␣` ile basılıyor.

Hücre Hugging Face Hub'dan iki tokenizer dosyası indirip `data/cache/huggingface` altında
önbelleğe alıyor: çok dilli BERT'inki (2 MB, Apache-2.0) ve XLM-R'ınki (9 MB, MIT).

In [14]:
from tokenizers import Tokenizer

two_words = "evlerimizden geldik"
gpt = tiktoken.get_encoding("cl100k_base")
xlmr = Tokenizer.from_pretrained("FacebookAI/xlm-roberta-base")
mbert = Tokenizer.from_pretrained("google-bert/bert-base-multilingual-cased")

rows = [
    ("GPT (cl100k_base)",
     [gpt.decode([t]).replace(" ", "␣") for t in gpt.encode(two_words)],
     "ev ler im iz den ␣geld ik"),
    ("SentencePiece (XLM-R)",
     xlmr.encode(two_words, add_special_tokens=False).tokens,
     "▁ev lerimiz den ▁geldi k"),
    ("WordPiece (multilingual BERT)",
     mbert.encode(two_words, add_special_tokens=False).tokens,
     "ev ##leri ##mi ##zde ##n geldi ##k"),
]
for family, tokens, book in rows:
    vs_book(family, " ".join(tokens), book, "{}")

GPT (cl100k_base)                      this run ev ler im iz den ␣geld ik   book ev ler im iz den ␣geld ik   same as the book
SentencePiece (XLM-R)                  this run ▁ev lerimiz den ▁geldi k   book ▁ev lerimiz den ▁geldi k   same as the book
WordPiece (multilingual BERT)          this run ev ##leri ##mi ##zde ##n geldi ##k   book ev ##leri ##mi ##zde ##n geldi ##k   same as the book


**Nasıl okunur.** GPT boşluğu *sonraki* kelimenin ilk token'ına yapıştırıyor (`␣geld`).
SentencePiece kelimenin başına görünür bir `▁` koyuyor. WordPiece tersini işaretliyor:
kelimenin *devamı* olan her parçanın başına `##` koyuyor. Üçü de aynı bilgiyi taşıyor:
burada yeni bir kelime başlıyor.

**Bir tokenizer'ın dört aşaması.** Metin modele gitmeden önce dört aşamadan geçer:
normalleştirme, ön tokenleştirme, alt kelime kodlaması ve özel token'lar. Bölüm Türkçe bir
cümleyi `bert-base-multilingual-uncased` tokenizer'ında (2 MB'lık bir dosya, Apache-2.0)
adım adım izliyor. Kod bölümün kodu.

**Neye bakacaksınız:** ilk satır; `Çay` orada `cay` oluyor.

In [15]:
record()
from tokenizers import Tokenizer

tok = Tokenizer.from_pretrained("google-bert/bert-base-multilingual-uncased")
text = "Çay içtik, sonra evlerimizden çıktık."

norm = tok.normalizer.normalize_str(text)
print("1 normalize    :", norm)
print("2 pre-tokenize :", [w for w, _ in tok.pre_tokenizer.pre_tokenize_str(norm)])
print("3 subwords     :", tok.encode(text, add_special_tokens=False).tokens)
print("4 special      :", tok.encode(text).tokens)
print("  ids          :", tok.encode(text).ids)
compare_with_book("""
1 normalize    : cay ictik, sonra evlerimizden cıktık.
2 pre-tokenize : ['cay', 'ictik', ',', 'sonra', 'evlerimizden', 'cıktık', '.']
3 subwords     : ['cay', 'ic', '##tik', ',', 'sonra', 'ev', '##leri', '##mi', '##zde', '##n', 'cıktı', '##k', '.']
4 special      : ['[CLS]', 'cay', 'ic', '##tik', ',', 'sonra', 'ev', '##leri', '##mi', '##zde', '##n', 'cıktı', '##k', '.', '[SEP]']
  ids          : [101, 21373, 11797, 13537, 117, 12624, 18110, 12737, 10555, 55567, 10115, 59161, 10167, 119, 102]
""")
print()
vs_book("the cased model leaves the text as it is", mbert.normalizer.normalize_str(text) == text, True, "{}")

1 normalize    : cay ictik, sonra evlerimizden cıktık.
2 pre-tokenize : ['cay', 'ictik', ',', 'sonra', 'evlerimizden', 'cıktık', '.']
3 subwords     : ['cay', 'ic', '##tik', ',', 'sonra', 'ev', '##leri', '##mi', '##zde', '##n', 'cıktı', '##k', '.']
4 special      : ['[CLS]', 'cay', 'ic', '##tik', ',', 'sonra', 'ev', '##leri', '##mi', '##zde', '##n', 'cıktı', '##k', '.', '[SEP]']
  ids          : [101, 21373, 11797, 13537, 117, 12624, 18110, 12737, 10555, 55567, 10115, 59161, 10167, 119, 102]



-> same as the book, line for line (5 lines)

the cased model leaves the text as it is this run      True   book      True   same as the book


**Nasıl okunur.** Her satır bir aşama. Normalleştirme metni küçük harfe çevirdi ve
harflerin üzerindeki işaretleri attı; model artık `çay` ile `cay` arasındaki farkı göremez.
`ı` yerinde kaldı, çünkü Unicode onu ayrı bir harf olarak tanımlıyor. Ön tokenleştirme
virgülü ve noktayı ayrı birer parça yaptı. Alt kelime kodlaması her kelimenin içinde
çalıştı: `evlerimizden` beş parça, `sonra` bütün kaldı. Son aşama `[CLS]` ve `[SEP]`
ekledi; tamsayılar modele gerçekten giden şey. Aynı modelin büyük/küçük harfe duyarlı
sürümü, bölümün dediği gibi, ilk aşamada metne dokunmuyor.

**Pratik bir ayrıntı: NFC.** `ö` gibi işaretli bir harf tek bir kod noktası olarak ya da
`o` harfi ile ayrı bir birleştirici işaret olarak saklanabilir. İkisi ekranda aynı görünür,
ama tokenizer onları farklı sayar. Bölümün önerisi bu yüzden tokenizer'dan önce tek bir
satır: `unicodedata.normalize("NFC", text)`. Bölüm bunun için sayı vermiyor; aşağıdakiler
bu koşunun, tek bir kelime üzerindeki sayıları.

In [16]:
shown = "görüşebileceklerimiz"
nfc, nfd = unicodedata.normalize("NFC", shown), unicodedata.normalize("NFD", shown)
print(f"NFC: {nfc}   {len(nfc)} code points, {len(nfc.encode('utf-8'))} bytes")
print(f"NFD: {nfd}   {len(nfd)} code points, {len(nfd.encode('utf-8'))} bytes")
print(f"equal in Python: {nfc == nfd}")
for name in ["cl100k_base", "o200k_base"]:
    e = tiktoken.get_encoding(name)
    print(f"{name:<12} NFC {len(e.encode(nfc)):>2} tokens   NFD {len(e.encode(nfd)):>2} tokens")
print(f"equal after normalize('NFC', ...): {unicodedata.normalize('NFC', nfd) == nfc}")

NFC: görüşebileceklerimiz   20 code points, 23 bytes
NFD: görüşebileceklerimiz   23 code points, 26 bytes
equal in Python: False
cl100k_base  NFC 10 tokens   NFD 15 tokens
o200k_base   NFC  7 tokens   NFD 10 tokens
equal after normalize('NFC', ...): True


**Nasıl okunur.** İki yazım aynı görünüyor, eşit karşılaştırılmıyor ve ayrıştırılmış olan
iki kodlamada da daha fazla token'a mal oluyor. Tek bir çağrı onları yeniden aynı metin
yapıyor.

**Neden doğrudan karakter kullanmıyoruz?** Bölümün sezgi kontrolü aynı Türkçe cümleyi üç
birim boyutunda sayıyor ve her sayının karesini alıyor; çünkü self-attention her birimi
diğer bütün birimlerle karşılaştırır. Kod bölümün kodu; `tr`, yukarıdaki hücredeki 1.
maddenin Türkçe metni.

**Neye bakacaksınız:** ortadaki sütun, sonra hemen yanındaki.

In [17]:
record()
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")
units = {
    "code points": list(tr),
    "subword tokens": [enc.decode([t]) for t in enc.encode(tr)],
    "words": tr.split(),
}
for name, seq in units.items():
    n = len(seq)
    print(f"{name:<15} {n:>3} units  {n * n:>6,} pairs  first six: {seq[:6]}")
compare_with_book("""
code points     160 units  25,600 pairs  first six: ['B', 'ü', 't', 'ü', 'n', ' ']
subword tokens   60 units   3,600 pairs  first six: ['B', 'üt', 'ün', ' insan', 'lar', ' h']
words            21 units     441 pairs  first six: ['Bütün', 'insanlar', 'hür,', 'haysiyet', 've', 'haklar']
""")

code points     160 units  25,600 pairs  first six: ['B', 'ü', 't', 'ü', 'n', ' ']
subword tokens   60 units   3,600 pairs  first six: ['B', 'üt', 'ün', ' insan', 'lar', ' h']
words            21 units     441 pairs  first six: ['Bütün', 'insanlar', 'hür,', 'haysiyet', 've', 'haklar']



-> same as the book, line for line (3 lines)


**Nasıl okunur.** Kod noktaları en uzun diziyi veriyor: 160 birim ve 25.600 çift. Bütün
kelimeler en kısasını veriyor, 21 birim; ama kelime düzeyinde bir sözlük hiç görmediği
biçimlerle karşılaşır. Alt kelimeler ikisinin arasında kalıyor. Sayılar bu cümleye ve bu
tokenizer'a ait.

## 4. Karakter, alt kelime, bütün kelime: tek bir örnek

[Bölümdeki kısım](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/08-on-egitim.html#sec-metin-veri-birimler). Küçük bir derlem, üç tokenizer. Derlem on
iki Türkçe gövdenin beşer biçiminden oluşuyor: 60 biçim ve 216 geçiş. İki test kelimesi,
`evlerden` ve `okullardan`, bunların arasında yok. Kod bölümün kodu; ilk kısımdaki
`pair_counts`, `merge` ve `apply_merges` fonksiyonlarını yeniden kullanıyor.

**Neye bakacaksınız:** her tokenizer'ın, görmediği iki kelime için kaç birime ihtiyaç
duyduğu.

In [18]:
record()
stems = {"ev": "e", "el": "e", "göl": "e", "köy": "e", "gün": "e", "yüz": "e",
         "okul": "a", "yol": "a", "kol": "a", "yıl": "a", "kız": "a", "masa": "a"}
corpus = {}
for stem, v in stems.items():          # v: eklerin alacağı ünlü
    for suffix, n in [("", 4), (f"l{v}r", 2), (f"d{v}", 4),
                      (f"d{v}n", 4), (f"l{v}rd{v}", 4)]:
        corpus[stem + suffix] = n      # ev, evler, evde, evden, evlerde, ...
test = ["evlerden", "okullardan"]      # iki kelime de derlemde yok

chars = sorted({c for word in corpus for c in word})
words = {tuple(w) + ("</w>",): n for w, n in corpus.items()}
merges = []
while len(chars) + 1 + len(merges) < 60:   # sözlük 60 sembole ulaşınca dur
    best, _ = pair_counts(words).most_common(1)[0]
    merges.append(best)
    words = merge(words, best)

tokenizers = {
    "whole words": (len(corpus), lambda w: [w if w in corpus else "<UNK>"]),
    "characters": (len(chars), lambda w: list(w)),
    "subwords (BPE)": (len(chars) + 1 + len(merges),
                       lambda w: list(apply_merges(w, merges))),
}
for name, (size, split) in tokenizers.items():
    pieces = [p for w in test for p in split(w)]
    print(f"{name:<15} vocabulary {size:>2}   {len(pieces):>2} units: {' '.join(pieces)}")
compare_with_book("""
whole words     vocabulary 60    2 units: <UNK> <UNK>
characters      vocabulary 18   18 units: e v l e r d e n o k u l l a r d a n
subwords (BPE)  vocabulary 60    6 units: ev ler den</w> okul lar dan</w>
""")
print()
vs_book("forms in the corpus", len(corpus), 60)
vs_book("occurrences in the corpus", sum(corpus.values()), 216)

whole words     vocabulary 60    2 units: <UNK> <UNK>
characters      vocabulary 18   18 units: e v l e r d e n o k u l l a r d a n
subwords (BPE)  vocabulary 60    6 units: ev ler den</w> okul lar dan</w>



-> same as the book, line for line (3 lines)

forms in the corpus                    this run        60   book        60   same as the book
occurrences in the corpus              this run       216   book       216   same as the book


**Nasıl okunur.** Bütün kelime tokenizer'ının 60 girdisi var ve yine de iki kez `<UNK>`
diyor. Karakter tokenizer'ına 18 sembol yetiyor ve hiçbir şey kaybolmuyor, ama iki kelimeye
18 birim harcıyor. Kelime listesiyle aynı 60 girdide durdurulan BPE ise iki kelimeyi altı
parçayla karşılıyor; bu parçaları başka kelimelerden öğrendi.

### Derinleşme: aynı derlemde WordPiece ve Unigram LM

Bölümün kapalı gelen derinleşme kutusu bu derlemde iki alt kelime öğrenicisi daha eğitiyor;
her biri birkaç on satır. Bunlar öğretim amaçlı gerçekleştirimler: buradaki WordPiece en
sık çifti değil, $\frac{\text{sayım}(ab)}{\text{sayım}(a)\,\text{sayım}(b)}$ puanı en
yüksek çifti birleştiriyor; Unigram LM ise bütün alt dizilerle başlayıp buduyor. Kod
bölümün kodu.

In [19]:
import math

def train_wordpiece(corpus, size):
    words = {tuple([w[0]] + ["##" + c for c in w[1:]]): n for w, n in corpus.items()}
    vocab = {s for symbols in words for s in symbols}
    first = None
    while len(vocab) < size:
        pairs, single = Counter(), Counter()
        for symbols, n in words.items():
            for s in symbols:
                single[s] += n
            for a, b in zip(symbols, symbols[1:]):
                pairs[(a, b)] += n
        score = {p: c / (single[p[0]] * single[p[1]]) for p, c in pairs.items()}
        best = max(score, key=lambda p: (score[p], p))
        first = first or (pairs, score)
        glued = best[0] + best[1].removeprefix("##")
        new = {}
        for symbols, n in words.items():
            out, i = [], 0
            while i < len(symbols):
                if symbols[i:i + 2] == best:
                    out.append(glued)
                    i += 2
                else:
                    out.append(symbols[i])
                    i += 1
            new[tuple(out)] = n
        words, vocab = new, vocab | {glued}
    return vocab, first

def wordpiece_split(word, vocab):          # açgözlü: önce bilinen en uzun parça
    pieces, start = [], 0
    while start < len(word):
        for end in range(len(word), start, -1):
            piece = word[start:end] if start == 0 else "##" + word[start:end]
            if piece in vocab:
                break
        else:
            return ["[UNK]"]
        pieces.append(piece)
        start = end
    return pieces

def best_split(word, logp):                # Viterbi: en olası bölünme
    best = [(0.0, [])] + [(-math.inf, None)] * len(word)
    for end in range(1, len(word) + 1):
        for start in range(end):
            piece = word[start:end]
            if piece in logp and best[start][1] is not None:
                score = best[start][0] + logp[piece]
                if score > best[end][0]:
                    best[end] = (score, best[start][1] + [piece])
    return best[-1]

def train_unigram(corpus, size):
    counts = Counter()                     # büyük başla: her alt dizi (substring)
    for w, n in corpus.items():
        for i in range(len(w)):
            for j in range(i + 1, len(w) + 1):
                counts[w[i:j]] += n
    start_size = len(counts)
    logprobs = lambda c: {p: math.log(k / sum(c.values())) for p, k in c.items()}
    loss = lambda lp: -sum(n * best_split(w, lp)[0] for w, n in corpus.items())
    while len(counts) > size:
        lp = logprobs(counts)
        base = loss(lp)
        cost = {p: loss({q: v for q, v in lp.items() if q != p}) - base
                for p in counts if len(p) > 1}     # tek harfler her zaman kalır
        k = max(1, min(len(counts) - size, len(cost) // 5))
        for p in sorted(cost, key=lambda p: (cost[p], p))[:k]:
            del counts[p]                  # en ucuza mal olan parçaları buda
        lp = logprobs(counts)              # en iyi bölünmelerden yeniden kestir
        counts = Counter({p: 0.001 for p in counts})
        for w, n in corpus.items():
            for piece in best_split(w, lp)[1]:
                counts[piece] += n
    return logprobs(counts), start_size

In [20]:
record()
wp_vocab, (pairs, score) = train_wordpiece(corpus, 60)
uni_logp, start_size = train_unigram(corpus, 60)

print("first step, WordPiece initial pairs:")
print("  highest raw pair counts:  ",
      [(a + b.removeprefix("##"), c) for (a, b), c in pairs.most_common(3)])
print("  highest score (WordPiece):",
      [(a + b.removeprefix("##"), round(s, 3))
       for (a, b), s in sorted(score.items(), key=lambda x: -x[1])[:3]])
print(f"  Unigram LM starts from {start_size} substrings and prunes to {len(uni_logp)}")
print()
for w in test:
    print(f"{w:<11} BPE       : {' | '.join(apply_merges(w, merges))}")
    print(f"{'':<11} WordPiece : {' | '.join(wordpiece_split(w, wp_vocab))}")
    print(f"{'':<11} Unigram LM: {' | '.join(best_split(w, uni_logp)[1])}")
compare_with_book("""
first step, WordPiece initial pairs:
  highest raw pair counts:   [('##de', 72), ('##da', 72), ('##rd', 48)]
  highest score (WordPiece): [('ok', 0.056), ('##ku', 0.056), ('ev', 0.028)]
  Unigram LM starts from 322 substrings and prunes to 60

evlerden    BPE       : ev | ler | den</w>
            WordPiece : ev | ##l | ##e | ##r | ##d | ##e | ##n
            Unigram LM: evlerde | n
okullardan  BPE       : okul | lar | dan</w>
            WordPiece : okullar | ##d | ##a | ##n
            Unigram LM: okullarda | n
""")

first step, WordPiece initial pairs:
  highest raw pair counts:   [('##de', 72), ('##da', 72), ('##rd', 48)]
  highest score (WordPiece): [('ok', 0.056), ('##ku', 0.056), ('ev', 0.028)]
  Unigram LM starts from 322 substrings and prunes to 60

evlerden    BPE       : ev | ler | den</w>
            WordPiece : ev | ##l | ##e | ##r | ##d | ##e | ##n
            Unigram LM: evlerde | n
okullardan  BPE       : okul | lar | dan</w>
            WordPiece : okullar | ##d | ##a | ##n
            Unigram LM: okullarda | n



-> same as the book, line for line (11 lines)


**Nasıl okunur.** İlk üç satır ölçütteki farkı gösteriyor: en sık geçen çiftler, WordPiece
puanı en yüksek çiftler değil. Üç öğrenici de aynı derlemden ve aynı sözlük boyutundan
yola çıkıp iki test kelimesini farklı bölüyor.

Unigram LM birleştirme saklamaz; her parça için bir olasılık saklar ve çarpımı en yüksek
bölünmeyi seçer. Sonraki hücre, bu sözlüğün `evlerden` için izin verdiği bütün bölünmeleri
puanlarıyla listeliyor.

In [21]:
record()
def every_split(word, logp, prefix=()):
    if not word:
        yield sum(logp[p] for p in prefix), list(prefix)
    for end in range(1, len(word) + 1):
        if word[:end] in logp:
            yield from every_split(word[end:], logp, prefix + (word[:end],))

splits = sorted(every_split("evlerden", uni_logp), reverse=True)
print(f"{len(splits)} ways to split 'evlerden' with this vocabulary:")
for logp, pieces in splits:
    print(f"  score = {math.exp(logp):.1e}   {' | '.join(pieces)}")
compare_with_book("""
4 ways to split 'evlerden' with this vocabulary:
  score = 2.1e-03   evlerde | n
  score = 1.2e-14   evler | d | e | n
  score = 1.3e-22   ev | l | e | r | d | e | n
  score = 1.1e-31   e | v | l | e | r | d | e | n
""")

4 ways to split 'evlerden' with this vocabulary:
  score = 2.1e-03   evlerde | n
  score = 1.2e-14   evler | d | e | n
  score = 1.3e-22   ev | l | e | r | d | e | n
  score = 1.1e-31   e | v | l | e | r | d | e | n



-> same as the book, line for line (5 lines)


**SentencePiece ile karşılaştırma.** Bölüm gerçek Unigram gerçekleştirimini aynı derlemde
eğitiyor ve ona aynı iki kelimeyi veriyor. Kod bölümün kodu, tek değişiklikle: dosyaları
çalışma dizini yerine `data/cache/` altına yazıyor. Hücre ardından bölümün bu koşu hakkında
söylediklerini kontrol ediyor.

In [22]:
record()
import sentencepiece as spm

lines = [w for w, n in corpus.items() for _ in range(n)]
mini = CACHE / "sentencepiece-mini"                 # kitapta yok: dosyaları önbellekte tut
mini.mkdir(parents=True, exist_ok=True)
with open(mini / "corpus.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n")

spm.SentencePieceTrainer.train(
    input=str(mini / "corpus.txt"), model_prefix=str(mini / "uni"), model_type="unigram", vocab_size=60,
    character_coverage=1.0, hard_vocab_limit=False, minloglevel=2,
    bos_id=-1, eos_id=-1, pad_id=-1)
uni = spm.SentencePieceProcessor(model_file=str(mini / "uni.model"))
for w in test:
    print(f"{w:<11} {' | '.join(uni.encode(w, out_type=str))}")
compare_with_book("""
evlerden    ▁evlerde | n
okullardan  ▁okul | l | a | r | d | a | n
""")
print()
pieces = [uni.id_to_piece(i) for i in range(uni.get_piece_size())]
vs_book("entries, including <unk>", len(pieces), 60)
vs_book("<unk> is one of them", "<unk>" in pieces, True, "{}")
vs_book("vocabulary contains ▁okul", "▁okul" in pieces, True, "{}")
vs_book("vocabulary contains ▁evlerde", "▁evlerde" in pieces, True, "{}")
vs_book("vocabulary contains ▁okullarda", "▁okullarda" in pieces, False, "{}")

evlerden    ▁evlerde | n
okullardan  ▁okul | l | a | r | d | a | n



-> same as the book, line for line (2 lines)

entries, including <unk>               this run        60   book        60   same as the book
<unk> is one of them                   this run      True   book      True   same as the book
vocabulary contains ▁okul              this run      True   book      True   same as the book
vocabulary contains ▁evlerde           this run      True   book      True   same as the book
vocabulary contains ▁okullarda         this run     False   book     False   same as the book


**Nasıl okunur.** SentencePiece ile öğretim amaçlı sürüm `evlerden` kelimesinde aynı
sonucu veriyor, diğer kelimenin ayrıntılarında ayrılıyor; eğitim yordamları aynı değil.
Bölümün bu kısmın tamamı için koyduğu çekince burada da geçerli: bunlar 216 kelime
geçişlik bir derlemde altmışar girdilik sözlükler; mekaniği göstermek için kuruldular,
algoritmaları karşılaştırmak için değil.

## 5. WordPiece ne zaman `[UNK]` der

[Mülakat soruları](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/08-on-egitim.html#sec-on-egitim-mulakat). Bölümün BPE üzerine mülakat cevabı, bir tokenizer'ın bilinmeyen
token üretip üretmeyeceğini yalnızca bayt düzeyinde bir alfabenin belirlemediğini not
ediyor: çok dilli BERT'in tokenizer'ında, gerçekleştirimin sınırından uzun bir kelime,
bütün karakterleri bilinse bile `[UNK]` oluyor. Bölüm kod vermiyor. **Bizim okuyuşumuz:**
üçüncü kısımdaki `bert-base-multilingual-cased` tokenizer dosyası; 100 ve 101 tane `a`
harfinden oluşan birer kelime üzerinde.

In [23]:
from tokenizers import Tokenizer

mbert = Tokenizer.from_pretrained("bert-base-multilingual-cased")
for n, book_pieces, book_first in [(100, 50, "aa"), (101, 1, "[UNK]")]:
    pieces = mbert.encode("a" * n, add_special_tokens=False).tokens
    vs_book(f"pieces for a word of {n} letters", len(pieces), book_pieces)
    vs_book("  the first piece", pieces[0], book_first, "{}")

pieces for a word of 100 letters       this run        50   book        50   same as the book
  the first piece                      this run        aa   book        aa   same as the book
pieces for a word of 101 letters       this run         1   book         1   same as the book
  the first piece                      this run     [UNK]   book     [UNK]   same as the book


## 6. Alıştırmalar

Bunlar [8. bölümün kendi alıştırmalarından](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/08-on-egitim.html#alıştırmalar) 11 ve 13 numaralı olanları yukarıdaki
hücrelerden başlayarak sürdürüyor.

1. **Tokenizer ekonomisi.** Üçüncü kısımdaki iki metni kendi yazdığınız teknik bir Türkçe
   paragrafla ve onun İngilizce çevirisiyle değiştirip `cl100k_base` ve `o200k_base`
   altında token sayın. Önce aynı içeriğin iki dildeki token sayısı oranını hesaplayın.
   Ardından seçtiğiniz bir modelin, tarihini belirttiğiniz birim fiyatlarıyla ayda 100.000
   istek için yıllık maliyeti kestirin; girdi ve çıktı token'larını ayrı bildirin ve tek
   paragrafın bütün istekleri temsil ettiği varsayımını yazın. Orana güvenmeden önce kendi
   kelimelerinizden birkaçını, `ev` hücresinin yaptığı gibi, önünde boşlukla ve boşluksuz
   sayın: boşluk cevabınızı değiştiriyor mu?

2. **Türkçe token maliyetini çok sayıda örnekte ölçün.** Üçüncü kısımdaki ölçüm tek bir kısa
   metin çifti üzerindeydi. Elinizdeki paralel bir metinden 50 ile 100 arası cümle çifti
   toplayın ve her tokenizer için iki ayrı ölçüm yapın: her çiftte Türkçe/İngilizce token
   sayısı oranı ve her dilde 1.000 Unicode kod noktası başına token sayısı. Medyanı ve
   çeyrekler arası aralığı bildirin; tek metin çiftindeki ölçümün bu dağılımın neresine
   düştüğünü söyleyin.

Kelime düzeyi token'lar, normalleştirme, Zipf ve Heaps ile TF-IDF
[ikinci bölümün notebook'unda](https://github.com/bbardakk/uygulamali-dogal-dil-isleme-notebooks/blob/main/tr/chapters/02-metin-veri/02-metin-veri.ipynb).

---

*Bu notebook'taki kod MIT lisanslı; anlatım CC BY 4.0. İki 1. madde metni, İnsan Hakları
Evrensel Bildirgesi'nin Birleşmiş Milletler tarafından yayımlanan resmî İngilizce ve Türkçe
sürümleri. Tokenizer sözlükleri `tiktoken` tarafından OpenAI'dan indiriliyor; çok dilli
BERT, XLM-R ve diğer modellerin tokenizer dosyaları Hugging Face Hub'dan indiriliyor ve
kendi lisanslarını koruyor.*